# RecursiveGCD (Python)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/RecursiveGCD_python.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `RecursiveGCD`: l'algoritmo in Python, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/RecursiveGCD_cpp.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/RecursiveGCD_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_python.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Python 3, senza installare nulla.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno i nomi dello pseudocodice. Due eccezioni in Python, dove `from` e `next` hanno un significato speciale: `Hanoi` usa `frm` al posto di `from` e `IterativeFib` usa `nxt` al posto di `next`.

## Strumenti: contatori, stack e traccia

La cella definisce i contatori (`cnt`, `calls`, `max_depth`), lo stack delle chiamate (`push` e `pop`) e le funzioni che stampano la traccia (`event`, `say`) e la tabella dei passi (`print_steps`). Basta eseguirla una volta.

In [1]:
cnt = [0] * 20      # cnt[r] = numero di esecuzioni della riga r (righe numerate come nelle dispense)
calls = 0           # numero di chiamate = nodi dell'albero delle chiamate
max_depth = 0       # profondita' della ricorsione = altezza massima dello stack
mults = 0           # moltiplicazioni (FastPower)
moves = 0           # mosse (Hanoi)
ev = 0              # contatore degli eventi della traccia
stack = []          # lo stack delle chiamate: gli argomenti delle chiamate attive
seen = [0] * 64     # seen[x] = numero di chiamate con argomento x
trace = False       # True: stampa la traccia dell'esecuzione


def reset():
    global calls, max_depth, mults, moves, ev
    calls = max_depth = mults = moves = ev = 0
    stack.clear()
    for r in range(len(cnt)):
        cnt[r] = 0
    for k in range(len(seen)):
        seen[k] = 0


def push(x):                       # inizia una chiamata: il suo frame va in cima allo stack
    global calls, max_depth
    calls += 1
    stack.append(x)
    max_depth = max(max_depth, len(stack))
    if x < len(seen):
        seen[x] += 1


def pop():                         # la chiamata termina: il suo frame esce dallo stack
    stack.pop()


def event(text):                   # riga numerata della traccia, con lo stack
    global ev
    ev += 1
    print("%2d. %s   stack: %s" % (ev, text, stack))


def say(call, action):             # riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
    print("%-28s| %s" % ("  " * (len(stack) - 1) + call, action))


def print_steps(*lines):
    print("Riga | Esecuzioni")
    for r in lines:
        print("%4d | %10d" % (r, cnt[r]))
    print("Passi totali S = %d" % sum(cnt))

## Massimo comun divisore: RecursiveGCD

**Input:** due numeri naturali $n \ge 1$ e $m \ge 0$. **Output:** $\gcd(n, m)$. Ricorsione di coda: ogni chiamata ne fa al più una, l'albero è un cammino. Al più $2\lfloor\log_2 m\rfloor + 3$ chiamate, $O(\log_2 m)$ passi.

Pseudocodice: sezione 14.6 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [2]:
def floor_log2(m):                                 # floor(log2 m) con un ciclo (solo per il limite sulle chiamate)
    k = 0
    while m > 1:
        m = m // 2
        k = k + 1
    return k


def recursive_gcd(n, m):
    push(m); cnt[2] += 1
    if m == 0:
        cnt[3] += 1
        if trace: say("RecursiveGCD(%d, %d)" % (n, m), "caso base, restituisce %d" % n)
        pop()
        return n
    if trace: say("RecursiveGCD(%d, %d)" % (n, m), "%d mod %d = %d: chiama RecursiveGCD(%d, %d)" % (n, m, n % m, m, n % m))
    cnt[5] += 1
    g = recursive_gcd(m, n % m)
    if trace: say("RecursiveGCD(%d, %d)" % (n, m), "restituisce %d" % g)
    pop()
    return g

In [3]:
# Esempi delle dispense: RecursiveGCD(48, 18) e RecursiveGCD(252, 105), alberi delle chiamate (cammini)
for n, m in [(48, 18), (252, 105)]:
    reset()
    trace = True
    print("%-28s| %s" % ("chiamata", "azione"))
    g = recursive_gcd(n, m)
    trace = False
    print("gcd(%d, %d) = %d, chiamate = %d, S = %d" % (n, m, g, calls, sum(cnt)))
    print()

# Numero di chiamate (ogni chiamata esegue 2 righe, quindi S = 2 chiamate) e limite 2 floor(log2 m) + 3
print("      n |       m |  gcd | chiamate |   S | 2 floor(log2 m) + 3")
for n, m in [(48, 18), (252, 105), (89, 55), (144, 89), (1000, 7), (1000000, 999999)]:
    reset()
    g = recursive_gcd(n, m)
    print("%7d | %7d | %4d | %8d | %3d | %d" % (n, m, g, calls, sum(cnt), 2 * floor_log2(m) + 3))

chiamata                    | azione
RecursiveGCD(48, 18)        | 48 mod 18 = 12: chiama RecursiveGCD(18, 12)
  RecursiveGCD(18, 12)      | 18 mod 12 = 6: chiama RecursiveGCD(12, 6)
    RecursiveGCD(12, 6)     | 12 mod 6 = 0: chiama RecursiveGCD(6, 0)
      RecursiveGCD(6, 0)    | caso base, restituisce 6
    RecursiveGCD(12, 6)     | restituisce 6
  RecursiveGCD(18, 12)      | restituisce 6
RecursiveGCD(48, 18)        | restituisce 6
gcd(48, 18) = 6, chiamate = 4, S = 8

chiamata                    | azione
RecursiveGCD(252, 105)      | 252 mod 105 = 42: chiama RecursiveGCD(105, 42)
  RecursiveGCD(105, 42)     | 105 mod 42 = 21: chiama RecursiveGCD(42, 21)
    RecursiveGCD(42, 21)    | 42 mod 21 = 0: chiama RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | caso base, restituisce 21
    RecursiveGCD(42, 21)    | restituisce 21
  RecursiveGCD(105, 42)     | restituisce 21
RecursiveGCD(252, 105)      | restituisce 21
gcd(252, 105) = 21, chiamate = 4, S = 8

      n |       m |  gcd | ch

**Prova tu.** Cambia i due numeri; il caso peggiore sono due numeri di Fibonacci consecutivi.

In [4]:
n, m = 1071, 462        # <-- cambia i due numeri
reset()
trace = True
print("%-28s| %s" % ("chiamata", "azione"))
g = recursive_gcd(n, m)
trace = False
print("gcd(%d, %d) = %d, chiamate = %d" % (n, m, g, calls))

chiamata                    | azione
RecursiveGCD(1071, 462)     | 1071 mod 462 = 147: chiama RecursiveGCD(462, 147)
  RecursiveGCD(462, 147)    | 462 mod 147 = 21: chiama RecursiveGCD(147, 21)
    RecursiveGCD(147, 21)   | 147 mod 21 = 0: chiama RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | caso base, restituisce 21
    RecursiveGCD(147, 21)   | restituisce 21
  RecursiveGCD(462, 147)    | restituisce 21
RecursiveGCD(1071, 462)     | restituisce 21
gcd(1071, 462) = 21, chiamate = 4


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).